# MAPS_integration_P

In [ ]:
import argparse
import datetime
import os
import os.path as osp
import random
import sys
import time
from pathlib import Path

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from MAPS.integration_P.datasets.datasets import LiverCancer
from MAPS.integration_P.model.network import Network, init_weights
from MAPS.integration_P.utils.utils import Logger
from MAPS.integration_P.utils.utils_dataloader import liver_dataloader
from MAPS.integration_P.utils.utils_training import train, test

## Configuration

In [ ]:
parser = argparse.ArgumentParser(description='Multi-omics translation')

################
# for model
parser.add_argument('--noise_rate', default=0.2, type=float)
parser.add_argument('--dropout_rate', default=0.1, type=float)

# Datasets
parser.add_argument('--path_omics', type=str, default='/mnt/sdf/2024_Xenium_HCC_Zhikang/Organized_xenium_pcf_h5ad',
                    help='path to the omics data (h5ad files)')
parser.add_argument('--path_img', type=str, default='/mnt/sdf/zhikangwang/fudan_Xenium_liver_cancer_organize/Image_features_registration_v2',
                    help='path to the morphology features (sequenced) (h5 files)')
parser.add_argument('--fea_dir', type=str, default='/mnt/sdf/zhikangwang/fudan_Xenium_liver_cancer_organize/align_seg_results_yunzhi_v2/Extracted_features', 
                    help='path to the morphology features (imaging-only) (h5 files)')

parser.add_argument('--hvg', type=int, default=500, 
                    help='number of highly variable genes')
parser.add_argument('--return_data', type=bool, default=True)

# Training options
parser.add_argument('--workers', default=16, type=int)
parser.add_argument('--max-epoch', default=45, type=int)
parser.add_argument('--stepsize', default=15, type=int)

parser.add_argument('--train-batch', default=2048, type=int)
parser.add_argument('--test-batch', default=2048, type=int) 

# Optimization options
parser.add_argument('--optimizer', default='adam', type=str,
                    help="adam or SGD")
parser.add_argument('--lr', '--learning-rate', default=0.001, type=float,
                    help="initial learning rate, use 0.0001")
parser.add_argument('--gamma', default=0.1, type=float,
                    help="learning rate decay")
parser.add_argument('--weight-decay', default=5e-04, type=float,
                    help="weight decay (default: 5e-04)")

# Miscs
parser.add_argument('--seed', type=int, default=1, 
                    help="manual seed")
parser.add_argument('--save-dir', type=str, default='./log', 
                    help="manual seed")
parser.add_argument('--eval-step', type=int, default=15, #
                    help="run evaluation for every N epochs (set to -1 to test after training)")
parser.add_argument('--gpu-devices', default='7', type=str, 
                    help='gpu device ids for CUDA_VISIBLE_DEVICES')

args = parser.parse_args(args=[])

## Pre-setting

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    return None

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

Path(args.save_dir).mkdir(parents=True, exist_ok=True)
Path('plots').mkdir(parents=True, exist_ok=True)

index = len(os.listdir(args.save_dir)) + 1
sys.stdout = Logger(os.path.join(args.save_dir, 'log_train_{}.txt'.format(str(index))))

print("==========\nArgs:{}\n==========".format(args))

## Create the dataloaders

In [ ]:
dataset = LiverCancer(path_omics = args.path_omics, path_img=args.path_img, hvg=args.hvg, return_data=args.return_data)
trainloader, testloader = liver_dataloader(args, dataset)

## Model initialization

In [ ]:
model = Network(omics1_size=dataset.omics1_size, 
                omics2_size=dataset.omics2_size, 
                noise_rate=args.noise_rate, 
                dropout_rate=args.dropout_rate)

model.apply(init_weights)
model = nn.DataParallel(model).cuda()

## Criterion and optimization

In [ ]:
criterion_mse, criterion_ce = nn.MSELoss(), nn.CrossEntropyLoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

## Model training and testing

In [ ]:
start_time = time.time()

fea_list = [sc.read_h5ad(os.path.join(args.fea_dir, f'{id}_fea.h5ad')).X for id in ['ID2', 'ID99'] ]

for epoch in range(args.max_epoch):
    ################
    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch), flush=True)

    train(model, criterion_mse, criterion_ce, optimizer, trainloader, fea_list)

    if args.stepsize > 0: scheduler.step()
    
    if (epoch+1) % args.eval_step == 0:
        print("==> Test", flush=True)
        test(model, testloader, dataset.omics1_panel, dataset.omics2_panel)
    ################

torch.save(model.state_dict(), 'last.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))